In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import math
import os
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


# =============================================================================
# Configuration
# =============================================================================


@dataclass(frozen=True)
class Config:
    seed: int = 42
    n_splits: int = 5

    # Kaggle paths. Override BASE_PATH in a notebook before calling main() if
    # needed.
    base_path: Path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    output_path: Path = Path("/kaggle/working")

    # Typewell matching features.
    typewell_search_min: int = -180
    typewell_search_max: int = 180
    typewell_search_step: int = 5
    typewell_probe_offsets: tuple[int, ...] = (-100, -50, -25, 0, 25, 50, 100)

    # Spatial KNN prior. Keep k modest; the rows are dense and nearby rows can be
    # strongly correlated.
    spatial_neighbors: int = 48
    spatial_eps: float = 1e-3

    # Postprocessing.
    smooth_windows: tuple[int, ...] = (5, 11, 21)
    use_viterbi: bool = True
    viterbi_range: int = 90
    viterbi_step: int = 6
    viterbi_gr_weight: float = 0.55
    viterbi_prior_sigma: float = 24.0
    viterbi_transition_sigma: float = 8.0

    # Blend search.
    blend_random_trials: int = 2500


CFG = Config()


# =============================================================================
# Small utilities
# =============================================================================


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_median(values: pd.Series | np.ndarray, default: float = 0.0) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return default
    return float(np.median(arr))


def robust_scale(values: np.ndarray, default: float = 1.0) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 4:
        return default
    med = np.median(values)
    mad = np.median(np.abs(values - med))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale < 1e-6:
        scale = np.std(values)
    if not np.isfinite(scale) or scale < 1e-6:
        scale = default
    return float(scale)


def weighted_mean(values: np.ndarray, distances: np.ndarray, eps: float) -> tuple[np.ndarray, np.ndarray]:
    weights = 1.0 / np.maximum(distances, eps)
    weights = weights / np.maximum(weights.sum(axis=1, keepdims=True), eps)
    mean = (values * weights).sum(axis=1)
    centered = values - mean[:, None]
    std = np.sqrt(np.maximum((weights * centered * centered).sum(axis=1), 0.0))
    return mean, std


def well_id_from_horizontal(path: Path) -> str:
    return path.name.replace("__horizontal_well.csv", "")


def well_id_from_typewell(path: Path) -> str:
    return path.name.split("__typewell")[0]


# =============================================================================
# Data loading
# =============================================================================


def read_horizontal_files(path: Path) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []
    files = sorted(path.glob("*__horizontal_well.csv"))
    for file in files:
        df = pd.read_csv(file)
        df["well_id"] = well_id_from_horizontal(file)
        rows.append(df)
    if not rows:
        raise FileNotFoundError(f"No horizontal well CSV files found in {path}")
    return pd.concat(rows, ignore_index=True)


def read_typewell_files(path: Path) -> dict[str, pd.DataFrame]:
    result: dict[str, pd.DataFrame] = {}
    for file in sorted(path.glob("*__typewell*.csv")):
        well_id = well_id_from_typewell(file)
        tw = pd.read_csv(file)
        if "TVT" not in tw.columns:
            continue
        if "GR" not in tw.columns:
            tw["GR"] = np.nan
        tw = tw.sort_values("TVT").drop_duplicates("TVT").reset_index(drop=True)
        tw["GR_interp"] = (
            tw["GR"]
            .astype(float)
            .interpolate(limit_direction="both")
            .fillna(safe_median(tw["GR"], 0.0))
        )
        result[well_id] = tw
    return result


def add_ids(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["well_id", "MD"]).reset_index(drop=True)
    df["depth_idx"] = df.groupby("well_id", sort=False).cumcount().astype(np.int32)
    df["id"] = df["well_id"].astype(str) + "_" + df["depth_idx"].astype(str)
    return df


# =============================================================================
# Feature engineering: anchors, trajectory, GR, typewell, self-known GR
# =============================================================================


def tail_slope(sub: pd.DataFrame, n: int) -> float:
    known = sub.loc[sub["TVT_input"].notna(), ["MD", "TVT_input"]].tail(n)
    if len(known) < 2:
        return np.nan
    md_span = known["MD"].iloc[-1] - known["MD"].iloc[0]
    if not np.isfinite(md_span) or abs(md_span) < 1e-6:
        return np.nan
    return float((known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) / md_span)


def add_anchor_features(df: pd.DataFrame) -> pd.DataFrame:
    if "TVT_input" not in df.columns:
        df["TVT_input"] = np.nan
    if "TVT" not in df.columns:
        df["TVT"] = np.nan

    known = df["TVT_input"].notna()
    df["has_tvt_input"] = known.astype(np.int8)

    for src, tmp in [
        ("depth_idx", "_known_idx"),
        ("MD", "_known_md"),
        ("TVT_input", "_known_tvt"),
        ("X", "_known_x"),
        ("Y", "_known_y"),
        ("Z", "_known_z"),
    ]:
        df[tmp] = df[src].where(known)

    g = df.groupby("well_id", sort=False)
    for src, dst in [
        ("_known_idx", "anchor_idx"),
        ("_known_md", "anchor_md"),
        ("_known_tvt", "anchor_tvt"),
        ("_known_x", "anchor_x"),
        ("_known_y", "anchor_y"),
        ("_known_z", "anchor_z"),
    ]:
        df[dst] = g[src].ffill()

    df["idx_after_anchor"] = df["depth_idx"] - df["anchor_idx"]
    df["md_after_anchor"] = df["MD"] - df["anchor_md"]
    df["x_after_anchor"] = df["X"] - df["anchor_x"]
    df["y_after_anchor"] = df["Y"] - df["anchor_y"]
    df["z_after_anchor"] = df["Z"] - df["anchor_z"]
    df["dist_after_anchor"] = np.sqrt(
        df["x_after_anchor"] ** 2 + df["y_after_anchor"] ** 2 + df["z_after_anchor"] ** 2
    )
    df["horizontal_after_anchor"] = np.sqrt(df["x_after_anchor"] ** 2 + df["y_after_anchor"] ** 2)

    prefix = (
        df.loc[known, ["well_id", "depth_idx", "MD", "TVT_input"]]
        .groupby("well_id", sort=False)
        .agg(
            first_known_idx=("depth_idx", "first"),
            first_known_md=("MD", "first"),
            first_known_tvt=("TVT_input", "first"),
            last_known_idx=("depth_idx", "last"),
            last_known_md=("MD", "last"),
            last_known_tvt=("TVT_input", "last"),
            known_tvt_count=("TVT_input", "size"),
        )
        .reset_index()
    )
    slopes = (
        df.groupby("well_id", sort=False)
        .apply(
            lambda s: pd.Series(
                {
                    "slope_last_5": tail_slope(s, 5),
                    "slope_last_20": tail_slope(s, 20),
                    "slope_last_75": tail_slope(s, 75),
                    "slope_last_250": tail_slope(s, 250),
                }
            )
        )
        .reset_index()
    )

    df = df.merge(prefix, on="well_id", how="left")
    df = df.merge(slopes, on="well_id", how="left")

    prefix_span = (df["last_known_md"] - df["first_known_md"]).replace(0, np.nan)
    df["slope_prefix"] = (df["last_known_tvt"] - df["first_known_tvt"]) / prefix_span
    df["slope_anchor"] = (
        df["slope_last_75"]
        .fillna(df["slope_last_250"])
        .fillna(df["slope_last_20"])
        .fillna(df["slope_prefix"])
        .fillna(0.0)
    )
    df["linear_tvt_anchor"] = df["anchor_tvt"] + df["slope_anchor"] * df["md_after_anchor"]
    df["linear_tvt_prefix"] = df["last_known_tvt"] + df["slope_prefix"] * (df["MD"] - df["last_known_md"])
    df["linear_tvt_anchor"] = df["linear_tvt_anchor"].fillna(df["linear_tvt_prefix"])

    df["known_fraction"] = df["known_tvt_count"] / np.maximum(df.groupby("well_id")["well_id"].transform("size"), 1)
    df["idx_after_ps_frac"] = df["idx_after_anchor"] / np.maximum(df.groupby("well_id")["well_id"].transform("size"), 1)

    drop_cols = [c for c in df.columns if c.startswith("_known_")]
    return df.drop(columns=drop_cols)


def add_trajectory_features(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("well_id", sort=False)
    df["well_rows"] = g["MD"].transform("size").astype(np.int32)
    df["rel_idx"] = df["depth_idx"] / np.maximum(df["well_rows"] - 1, 1)

    for col in ["MD", "X", "Y", "Z"]:
        first = g[col].transform("first")
        last = g[col].transform("last")
        df[f"{col}_from_start"] = df[col] - first
        df[f"{col}_to_end"] = last - df[col]
        df[f"{col}_span"] = last - first
        df[f"d_{col}"] = g[col].diff()
        df[f"dd_{col}"] = g[f"d_{col}"].diff()

    dmd = df["d_MD"].replace(0, np.nan)
    df["dX_dMD"] = df["d_X"] / dmd
    df["dY_dMD"] = df["d_Y"] / dmd
    df["dZ_dMD"] = df["d_Z"] / dmd

    df["horizontal_step"] = np.sqrt(df["d_X"] ** 2 + df["d_Y"] ** 2)
    df["step_3d"] = np.sqrt(df["d_X"] ** 2 + df["d_Y"] ** 2 + df["d_Z"] ** 2)
    df["horizontal_per_md"] = df["horizontal_step"] / dmd.abs()
    df["step3d_per_md"] = df["step_3d"] / dmd.abs()
    df["curvature_proxy"] = np.sqrt(df["dd_X"] ** 2 + df["dd_Y"] ** 2 + df["dd_Z"] ** 2)

    df["cum_horizontal"] = g["horizontal_step"].cumsum()
    df["cum_3d"] = g["step_3d"].cumsum()
    direct_from_start = np.sqrt(df["X_from_start"] ** 2 + df["Y_from_start"] ** 2 + df["Z_from_start"] ** 2)
    df["tortuosity_from_start"] = df["cum_3d"] / np.maximum(direct_from_start, 1e-3)

    total_dx = g["X"].transform("last") - g["X"].transform("first")
    total_dy = g["Y"].transform("last") - g["Y"].transform("first")
    df["well_azimuth"] = np.arctan2(total_dy, total_dx)
    df["well_azimuth_sin"] = np.sin(df["well_azimuth"])
    df["well_azimuth_cos"] = np.cos(df["well_azimuth"])
    df["local_azimuth"] = np.arctan2(df["d_Y"], df["d_X"])
    df["local_azimuth_sin"] = np.sin(df["local_azimuth"])
    df["local_azimuth_cos"] = np.cos(df["local_azimuth"])
    df["anchor_azimuth"] = np.arctan2(df["y_after_anchor"], df["x_after_anchor"])
    df["anchor_azimuth_sin"] = np.sin(df["anchor_azimuth"])
    df["anchor_azimuth_cos"] = np.cos(df["anchor_azimuth"])

    for window in [11, 51, 151]:
        df[f"curvature_mean_{window}"] = g["curvature_proxy"].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean()
        )
        df[f"step3d_mean_{window}"] = g["step_3d"].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean()
        )
        df[f"dZ_dMD_mean_{window}"] = g["dZ_dMD"].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean()
        )

    return df


def add_gr_features(df: pd.DataFrame) -> pd.DataFrame:
    if "GR" not in df.columns:
        df["GR"] = np.nan

    global_gr = safe_median(df["GR"], 0.0)
    g = df.groupby("well_id", sort=False)
    df["GR_missing"] = df["GR"].isna().astype(np.int8)
    df["GR_interp"] = g["GR"].transform(lambda s: s.astype(float).interpolate(limit_direction="both")).fillna(global_gr)
    df["GR_centered_well"] = df["GR_interp"] - g["GR_interp"].transform("mean")
    df["GR_std_well_value"] = g["GR_interp"].transform("std").replace(0, np.nan)
    df["GR_z_well"] = df["GR_centered_well"] / df["GR_std_well_value"]
    df["GR_rank_well"] = g["GR_interp"].rank(pct=True)

    for window in [5, 21, 51, 101, 201]:
        roll = g["GR_interp"].transform(lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean())
        df[f"GR_mean_{window}"] = roll
        df[f"GR_delta_mean_{window}"] = df["GR_interp"] - roll
        df[f"GR_std_{window}"] = g["GR_interp"].transform(
            lambda s, w=window: s.rolling(w, min_periods=2, center=True).std()
        )
        df[f"GR_min_{window}"] = g["GR_interp"].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).min()
        )
        df[f"GR_max_{window}"] = g["GR_interp"].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).max()
        )

    df["d_GR"] = g["GR_interp"].diff()
    df["dd_GR"] = g["d_GR"].diff()
    df["dGR_dMD"] = df["d_GR"] / df["d_MD"].replace(0, np.nan)

    return df


def interp_typewell_gr(tw: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    tvt_axis = tw["TVT"].to_numpy(float)
    gr_axis = tw["GR_interp"].to_numpy(float)
    if len(tvt_axis) < 2:
        return np.full_like(tvt_values, np.nan, dtype=float)
    return np.interp(tvt_values, tvt_axis, gr_axis, left=np.nan, right=np.nan)


def add_typewell_features(df: pd.DataFrame, typewells: dict[str, pd.DataFrame], cfg: Config) -> pd.DataFrame:
    new_cols: dict[str, np.ndarray] = {}
    base_cols = ["linear_tvt_anchor", "linear_tvt_prefix", "anchor_tvt"]

    for col in base_cols:
        new_cols[f"tw_gr_at_{col}"] = np.full(len(df), np.nan, dtype=np.float32)
        new_cols[f"tw_gr_diff_{col}"] = np.full(len(df), np.nan, dtype=np.float32)

    for off in cfg.typewell_probe_offsets:
        new_cols[f"tw_gr_linear_off_{off:+d}"] = np.full(len(df), np.nan, dtype=np.float32)

    for name in ["wide", "near"]:
        new_cols[f"tw_best_offset_{name}"] = np.full(len(df), np.nan, dtype=np.float32)
        new_cols[f"tw_best_tvt_{name}"] = np.full(len(df), np.nan, dtype=np.float32)
        new_cols[f"tw_best_absdiff_{name}"] = np.full(len(df), np.nan, dtype=np.float32)
        new_cols[f"tw_best_gr_{name}"] = np.full(len(df), np.nan, dtype=np.float32)

    for well_id, idx in df.groupby("well_id", sort=False).groups.items():
        tw = typewells.get(well_id)
        if tw is None or tw.empty:
            continue
        idx_arr = np.asarray(idx)
        sub = df.loc[idx_arr]
        gr_obs = sub["GR_interp"].to_numpy(float)

        for col in base_cols:
            tvt_values = sub[col].to_numpy(float)
            tw_gr = interp_typewell_gr(tw, tvt_values)
            new_cols[f"tw_gr_at_{col}"][idx_arr] = tw_gr.astype(np.float32)
            new_cols[f"tw_gr_diff_{col}"][idx_arr] = (gr_obs - tw_gr).astype(np.float32)

        linear = sub["linear_tvt_anchor"].to_numpy(float)
        for off in cfg.typewell_probe_offsets:
            tw_gr = interp_typewell_gr(tw, linear + off)
            new_cols[f"tw_gr_linear_off_{off:+d}"][idx_arr] = tw_gr.astype(np.float32)

        for name, offsets in {
            "wide": np.arange(cfg.typewell_search_min, cfg.typewell_search_max + 1, cfg.typewell_search_step),
            "near": np.arange(-60, 61, cfg.typewell_search_step),
        }.items():
            best_cost = np.full(len(sub), np.inf, dtype=float)
            best_offset = np.full(len(sub), np.nan, dtype=float)
            best_gr = np.full(len(sub), np.nan, dtype=float)
            for off in offsets:
                candidate_gr = interp_typewell_gr(tw, linear + off)
                cost = np.abs(gr_obs - candidate_gr)
                valid = np.isfinite(cost) & (cost < best_cost)
                best_cost[valid] = cost[valid]
                best_offset[valid] = off
                best_gr[valid] = candidate_gr[valid]
            new_cols[f"tw_best_offset_{name}"][idx_arr] = best_offset.astype(np.float32)
            new_cols[f"tw_best_tvt_{name}"][idx_arr] = (linear + best_offset).astype(np.float32)
            new_cols[f"tw_best_absdiff_{name}"][idx_arr] = best_cost.astype(np.float32)
            new_cols[f"tw_best_gr_{name}"][idx_arr] = best_gr.astype(np.float32)

    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def add_self_known_gr_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    cols = {
        "self_gr_at_linear": np.full(len(df), np.nan, dtype=np.float32),
        "self_gr_diff_linear": np.full(len(df), np.nan, dtype=np.float32),
        "self_best_offset": np.full(len(df), np.nan, dtype=np.float32),
        "self_best_tvt": np.full(len(df), np.nan, dtype=np.float32),
        "self_best_absdiff": np.full(len(df), np.nan, dtype=np.float32),
    }

    offsets = np.arange(-120, 121, cfg.typewell_search_step)
    for _, idx in df.groupby("well_id", sort=False).groups.items():
        idx_arr = np.asarray(idx)
        sub = df.loc[idx_arr]
        known = sub["TVT_input"].notna() & sub["GR_interp"].notna()
        known_sub = sub.loc[known, ["TVT_input", "GR_interp"]].sort_values("TVT_input").drop_duplicates("TVT_input")
        if len(known_sub) < 8:
            continue

        known_tvt = known_sub["TVT_input"].to_numpy(float)
        known_gr = known_sub["GR_interp"].to_numpy(float)
        linear = sub["linear_tvt_anchor"].to_numpy(float)
        gr_obs = sub["GR_interp"].to_numpy(float)

        gr_at_linear = np.interp(linear, known_tvt, known_gr, left=np.nan, right=np.nan)
        cols["self_gr_at_linear"][idx_arr] = gr_at_linear.astype(np.float32)
        cols["self_gr_diff_linear"][idx_arr] = (gr_obs - gr_at_linear).astype(np.float32)

        best_cost = np.full(len(sub), np.inf, dtype=float)
        best_offset = np.full(len(sub), np.nan, dtype=float)
        for off in offsets:
            candidate_gr = np.interp(linear + off, known_tvt, known_gr, left=np.nan, right=np.nan)
            cost = np.abs(gr_obs - candidate_gr)
            valid = np.isfinite(cost) & (cost < best_cost)
            best_cost[valid] = cost[valid]
            best_offset[valid] = off

        cols["self_best_offset"][idx_arr] = best_offset.astype(np.float32)
        cols["self_best_tvt"][idx_arr] = (linear + best_offset).astype(np.float32)
        cols["self_best_absdiff"][idx_arr] = best_cost.astype(np.float32)

    return pd.concat([df, pd.DataFrame(cols, index=df.index)], axis=1)


def engineer(df: pd.DataFrame, typewells: dict[str, pd.DataFrame], cfg: Config) -> pd.DataFrame:
    df = add_ids(df.copy())
    df = add_anchor_features(df)
    df = add_trajectory_features(df)
    df = add_gr_features(df)
    df = add_typewell_features(df, typewells, cfg)
    df = add_self_known_gr_features(df, cfg)
    return df


# =============================================================================
# Spatial OOF prior
# =============================================================================


def spatial_feature_columns(frame: pd.DataFrame) -> list[str]:
    cols = [
        "X",
        "Y",
        "Z",
        "MD",
        "rel_idx",
        "md_after_anchor",
        "horizontal_after_anchor",
        "dist_after_anchor",
        "linear_tvt_anchor",
        "slope_anchor",
        "well_azimuth_sin",
        "well_azimuth_cos",
        "GR_interp",
        "GR_z_well",
        "tw_best_offset_wide",
        "tw_best_absdiff_wide",
    ]
    return [c for c in cols if c in frame.columns]


def fit_predict_spatial_prior(
    train_fit: pd.DataFrame,
    test_fit: pd.DataFrame,
    groups: pd.Series,
    cfg: Config,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_fit = train_fit.copy()
    test_fit = test_fit.copy()

    cols = spatial_feature_columns(train_fit)
    y_resid = (train_fit["TVT"] - train_fit["linear_tvt_anchor"]).to_numpy(float)
    y_abs = train_fit["TVT"].to_numpy(float)

    oof_resid = np.full(len(train_fit), np.nan, dtype=np.float32)
    oof_abs = np.full(len(train_fit), np.nan, dtype=np.float32)
    oof_std = np.full(len(train_fit), np.nan, dtype=np.float32)
    oof_min_dist = np.full(len(train_fit), np.nan, dtype=np.float32)

    gkf = GroupKFold(n_splits=cfg.n_splits)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_fit, y_resid, groups), 1):
        tr = train_fit.iloc[tr_idx]
        va = train_fit.iloc[va_idx]
        scaler = StandardScaler()
        med = tr[cols].median().fillna(0.0)
        x_tr = scaler.fit_transform(tr[cols].replace([np.inf, -np.inf], np.nan).fillna(med))
        x_va = scaler.transform(va[cols].replace([np.inf, -np.inf], np.nan).fillna(med))

        k = min(cfg.spatial_neighbors, len(tr_idx))
        nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
        nn.fit(x_tr)
        distances, neighbor_idx = nn.kneighbors(x_va)
        mean_resid, std_resid = weighted_mean(y_resid[tr_idx][neighbor_idx], distances, cfg.spatial_eps)
        mean_abs, _ = weighted_mean(y_abs[tr_idx][neighbor_idx], distances, cfg.spatial_eps)

        oof_resid[va_idx] = mean_resid.astype(np.float32)
        oof_abs[va_idx] = mean_abs.astype(np.float32)
        oof_std[va_idx] = std_resid.astype(np.float32)
        oof_min_dist[va_idx] = distances[:, 0].astype(np.float32)
        print(f"Spatial prior fold {fold} complete")

    scaler = StandardScaler()
    med = train_fit[cols].median().fillna(0.0)
    x_all = scaler.fit_transform(train_fit[cols].replace([np.inf, -np.inf], np.nan).fillna(med))
    x_test = scaler.transform(test_fit[cols].replace([np.inf, -np.inf], np.nan).fillna(med))
    k = min(cfg.spatial_neighbors, len(train_fit))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(x_all)
    distances, neighbor_idx = nn.kneighbors(x_test)
    test_resid, test_std = weighted_mean(y_resid[neighbor_idx], distances, cfg.spatial_eps)
    test_abs, _ = weighted_mean(y_abs[neighbor_idx], distances, cfg.spatial_eps)

    train_fit["spatial_resid_idw"] = oof_resid
    train_fit["spatial_abs_tvt_idw"] = oof_abs
    train_fit["spatial_resid_std_idw"] = oof_std
    train_fit["spatial_min_dist"] = oof_min_dist

    test_fit["spatial_resid_idw"] = test_resid.astype(np.float32)
    test_fit["spatial_abs_tvt_idw"] = test_abs.astype(np.float32)
    test_fit["spatial_resid_std_idw"] = test_std.astype(np.float32)
    test_fit["spatial_min_dist"] = distances[:, 0].astype(np.float32)

    return train_fit, test_fit


# =============================================================================
# Models and validation
# =============================================================================


def feature_columns(train_fit: pd.DataFrame, test_fit: pd.DataFrame) -> list[str]:
    blocked = {
        "TVT",
        "TVT_input",
        "id",
        "well_id",
        "Geology",
        "depth_idx",
    }
    common = sorted(set(train_fit.columns).intersection(test_fit.columns))
    cols: list[str] = []
    for col in common:
        if col in blocked:
            continue
        if pd.api.types.is_numeric_dtype(train_fit[col]) and pd.api.types.is_numeric_dtype(test_fit[col]):
            cols.append(col)
    return cols


def make_lgbm(seed: int, objective: str = "regression"):
    try:
        from lightgbm import LGBMRegressor

        return LGBMRegressor(
            objective=objective,
            n_estimators=5000,
            learning_rate=0.018,
            num_leaves=191,
            max_depth=-1,
            min_child_samples=55,
            subsample=0.88,
            subsample_freq=1,
            colsample_bytree=0.82,
            reg_alpha=0.08,
            reg_lambda=0.70,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        )
    except Exception:
        return HistGradientBoostingRegressor(
            max_iter=1600,
            learning_rate=0.025,
            max_leaf_nodes=95,
            min_samples_leaf=35,
            l2_regularization=0.08,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=seed,
        )


def fit_predict_model(model, x_train, y_train, x_valid, y_valid, x_test):
    if model.__class__.__name__ == "LGBMRegressor":
        import lightgbm as lgb

        model.fit(
            x_train,
            y_train,
            eval_set=[(x_valid, y_valid)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(220), lgb.log_evaluation(300)],
        )
    else:
        model.fit(x_train, y_train)
    return model.predict(x_valid), model.predict(x_test)


def train_cv_models(
    train_fit: pd.DataFrame,
    test_fit: pd.DataFrame,
    features: list[str],
    cfg: Config,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_fit = train_fit.copy()
    test_fit = test_fit.copy()

    x = train_fit[features].replace([np.inf, -np.inf], np.nan).astype(np.float32)
    x_test = test_fit[features].replace([np.inf, -np.inf], np.nan).astype(np.float32)
    groups = train_fit["well_id"]

    y_resid = (train_fit["TVT"] - train_fit["linear_tvt_anchor"]).astype(np.float32)
    y_direct = train_fit["TVT"].astype(np.float32)

    oof_resid = np.zeros(len(train_fit), dtype=np.float32)
    oof_direct = np.zeros(len(train_fit), dtype=np.float32)
    test_resid = np.zeros(len(test_fit), dtype=np.float32)
    test_direct = np.zeros(len(test_fit), dtype=np.float32)

    gkf = GroupKFold(n_splits=cfg.n_splits)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(x, y_resid, groups), 1):
        print(f"\nModel fold {fold}")

        model_resid = make_lgbm(cfg.seed + fold)
        pred_va_resid, pred_test_resid = fit_predict_model(
            model_resid,
            x.iloc[tr_idx],
            y_resid.iloc[tr_idx],
            x.iloc[va_idx],
            y_resid.iloc[va_idx],
            x_test,
        )
        oof_resid[va_idx] = pred_va_resid.astype(np.float32)
        test_resid += pred_test_resid.astype(np.float32) / cfg.n_splits

        model_direct = make_lgbm(cfg.seed + 100 + fold)
        pred_va_direct, pred_test_direct = fit_predict_model(
            model_direct,
            x.iloc[tr_idx],
            y_direct.iloc[tr_idx],
            x.iloc[va_idx],
            y_direct.iloc[va_idx],
            x_test,
        )
        oof_direct[va_idx] = pred_va_direct.astype(np.float32)
        test_direct += pred_test_direct.astype(np.float32) / cfg.n_splits

        fold_pred = train_fit["linear_tvt_anchor"].iloc[va_idx].to_numpy(float) + pred_va_resid
        print(f"Fold {fold} residual-model RMSE: {rmse(train_fit['TVT'].iloc[va_idx], fold_pred):.5f}")
        print(f"Fold {fold} direct-model RMSE:   {rmse(train_fit['TVT'].iloc[va_idx], pred_va_direct):.5f}")

    train_fit["pred_resid_lgbm"] = train_fit["linear_tvt_anchor"].to_numpy(float) + oof_resid
    train_fit["pred_direct_lgbm"] = oof_direct
    train_fit["pred_spatial"] = train_fit["linear_tvt_anchor"].to_numpy(float) + train_fit["spatial_resid_idw"].to_numpy(float)
    train_fit["pred_typewell_best"] = train_fit["tw_best_tvt_wide"].fillna(train_fit["linear_tvt_anchor"])

    test_fit["pred_resid_lgbm"] = test_fit["linear_tvt_anchor"].to_numpy(float) + test_resid
    test_fit["pred_direct_lgbm"] = test_direct
    test_fit["pred_spatial"] = test_fit["linear_tvt_anchor"].to_numpy(float) + test_fit["spatial_resid_idw"].to_numpy(float)
    test_fit["pred_typewell_best"] = test_fit["tw_best_tvt_wide"].fillna(test_fit["linear_tvt_anchor"])

    print("\nOOF residual-model RMSE:", rmse(train_fit["TVT"], train_fit["pred_resid_lgbm"]))
    print("OOF direct-model RMSE:  ", rmse(train_fit["TVT"], train_fit["pred_direct_lgbm"]))
    print("OOF spatial-prior RMSE: ", rmse(train_fit["TVT"], train_fit["pred_spatial"]))
    print("OOF typewell-best RMSE: ", rmse(train_fit["TVT"], train_fit["pred_typewell_best"]))

    return train_fit, test_fit


# =============================================================================
# Postprocessing and blending
# =============================================================================


def smooth_by_well(frame: pd.DataFrame, value_col: str, window: int, method: str = "median") -> np.ndarray:
    if window <= 1:
        return frame[value_col].to_numpy(float)
    if method == "median":
        smoothed = frame.groupby("well_id", sort=False)[value_col].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).median()
        )
    else:
        smoothed = frame.groupby("well_id", sort=False)[value_col].transform(
            lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean()
        )
    return smoothed.to_numpy(float)


def viterbi_path_for_well(sub: pd.DataFrame, tw: pd.DataFrame | None, pred_col: str, cfg: Config) -> np.ndarray:
    raw = sub[pred_col].to_numpy(float)
    if tw is None or len(sub) < 2 or len(tw) < 2 or "GR_interp" not in sub.columns:
        return raw

    gr_obs = sub["GR_interp"].to_numpy(float)
    if np.isfinite(gr_obs).sum() < 3:
        return raw

    offsets = np.arange(-cfg.viterbi_range, cfg.viterbi_range + 1, cfg.viterbi_step, dtype=float)
    states = raw[:, None] + offsets[None, :]
    state_count = states.shape[1]

    tw_gr = interp_typewell_gr(tw, states.reshape(-1)).reshape(states.shape)
    gr_sigma = robust_scale(np.concatenate([gr_obs[np.isfinite(gr_obs)], tw["GR_interp"].to_numpy(float)]), default=20.0)

    diff = (gr_obs[:, None] - tw_gr) / gr_sigma
    emission = cfg.viterbi_gr_weight * np.minimum(diff * diff, 9.0)
    emission[~np.isfinite(emission)] = 0.0

    prior_cost = (offsets[None, :] / cfg.viterbi_prior_sigma) ** 2
    costs = emission + prior_cost

    dp = np.zeros_like(costs, dtype=np.float64)
    back = np.zeros((len(sub), state_count), dtype=np.int16)
    dp[0] = costs[0]

    for i in range(1, len(sub)):
        expected_delta = raw[i] - raw[i - 1]
        transition = (
            (states[i - 1, :, None] + expected_delta - states[i, None, :])
            / cfg.viterbi_transition_sigma
        ) ** 2
        total = dp[i - 1, :, None] + transition
        back[i] = np.argmin(total, axis=0).astype(np.int16)
        dp[i] = costs[i] + total[back[i], np.arange(state_count)]

    path_idx = np.zeros(len(sub), dtype=np.int16)
    path_idx[-1] = int(np.argmin(dp[-1]))
    for i in range(len(sub) - 1, 0, -1):
        path_idx[i - 1] = back[i, path_idx[i]]

    return states[np.arange(len(sub)), path_idx]


def add_postprocessed_candidates(
    train_fit: pd.DataFrame,
    test_fit: pd.DataFrame,
    typewells: dict[str, pd.DataFrame],
    cfg: Config,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train_fit = train_fit.copy()
    test_fit = test_fit.copy()

    candidate_cols = [
        "linear_tvt_anchor",
        "pred_spatial",
        "pred_typewell_best",
        "pred_resid_lgbm",
        "pred_direct_lgbm",
    ]

    for base in ["pred_resid_lgbm", "pred_direct_lgbm"]:
        for window in cfg.smooth_windows:
            col = f"{base}_med{window}"
            train_fit[col] = smooth_by_well(train_fit, base, window, method="median")
            test_fit[col] = smooth_by_well(test_fit, base, window, method="median")
            candidate_cols.append(col)

    if cfg.use_viterbi:
        for frame, name in [(train_fit, "train"), (test_fit, "test")]:
            out = np.full(len(frame), np.nan, dtype=np.float32)
            for well_id, idx in frame.groupby("well_id", sort=False).groups.items():
                sub = frame.loc[idx]
                tw = typewells.get(well_id)
                path = viterbi_path_for_well(sub, tw, "pred_resid_lgbm", cfg)
                out[np.asarray(idx)] = path.astype(np.float32)
            frame["pred_viterbi"] = out
            print(f"Viterbi postprocess complete for {name}")
        candidate_cols.append("pred_viterbi")

    # De-duplicate while preserving order.
    candidate_cols = list(dict.fromkeys([c for c in candidate_cols if c in train_fit.columns and c in test_fit.columns]))
    return train_fit, test_fit, candidate_cols


def optimize_blend(train_fit: pd.DataFrame, candidate_cols: list[str], cfg: Config) -> tuple[np.ndarray, list[str]]:
    y = train_fit["TVT"].to_numpy(float)
    preds = train_fit[candidate_cols].replace([np.inf, -np.inf], np.nan)
    preds = preds.fillna(preds.median()).to_numpy(float)

    scores = {}
    for i, col in enumerate(candidate_cols):
        scores[col] = rmse(y, preds[:, i])
    print("\nCandidate OOF RMSE:")
    for col, score in sorted(scores.items(), key=lambda x: x[1]):
        print(f"  {col:28s} {score:.5f}")

    # Keep the search stable by blending only the strongest candidates.
    keep = [col for col, _ in sorted(scores.items(), key=lambda x: x[1])[: min(6, len(candidate_cols))]]
    keep_idx = [candidate_cols.index(c) for c in keep]
    p = preds[:, keep_idx]

    rng = np.random.default_rng(cfg.seed)
    best_w = np.zeros(len(keep), dtype=float)
    best_w[0] = 1.0
    best_score = rmse(y, p @ best_w)

    # Include a few deterministic starting points.
    starts = [np.eye(len(keep))[i] for i in range(len(keep))]
    starts.append(np.ones(len(keep)) / len(keep))
    for w in starts:
        score = rmse(y, p @ w)
        if score < best_score:
            best_score = score
            best_w = w.copy()

    for _ in range(cfg.blend_random_trials):
        alpha = np.ones(len(keep)) * 1.5
        w = rng.dirichlet(alpha)
        score = rmse(y, p @ w)
        if score < best_score:
            best_score = score
            best_w = w

    print("\nBest blend RMSE:", round(best_score, 5))
    print("Best blend weights:")
    for col, w in zip(keep, best_w):
        print(f"  {col:28s} {w:.4f}")

    return best_w, keep


def apply_blend(frame: pd.DataFrame, weights: np.ndarray, cols: list[str]) -> np.ndarray:
    preds = frame[cols].replace([np.inf, -np.inf], np.nan)
    preds = preds.fillna(preds.median()).to_numpy(float)
    return preds @ weights


# =============================================================================
# Output
# =============================================================================


def make_submission(sample_submission: pd.DataFrame, test_fit: pd.DataFrame, pred_col: str, path: Path) -> pd.DataFrame:
    out = sample_submission[["id"]].merge(test_fit[["id", pred_col]], on="id", how="left")
    out = out.rename(columns={pred_col: "tvt"})
    out["tvt"] = out["tvt"].astype(float)
    out["tvt"] = out["tvt"].fillna(out["tvt"].median())
    out.to_csv(path, index=False)
    print(f"Saved {path}")
    return out


def main(cfg: Config = CFG) -> None:
    np.random.seed(cfg.seed)
    train_path = cfg.base_path / "train"
    test_path = cfg.base_path / "test"
    sample_path = cfg.base_path / "sample_submission.csv"

    print("Base path:", cfg.base_path)
    print("Reading data")
    train = read_horizontal_files(train_path)
    test = read_horizontal_files(test_path)
    train_typewells = read_typewell_files(train_path)
    test_typewells = read_typewell_files(test_path)
    typewells = {**train_typewells, **test_typewells}
    sample_submission = pd.read_csv(sample_path)[["id"]]

    print("Train horizontal shape:", train.shape)
    print("Test horizontal shape:", test.shape)
    print("Train typewells:", len(train_typewells))
    print("Test typewells:", len(test_typewells))
    print("Sample rows:", len(sample_submission))

    print("\nEngineering features")
    train_feat = engineer(train, typewells, cfg)
    test_feat = engineer(test, typewells, cfg)

    train_mask = (
        train_feat["TVT"].notna()
        & train_feat["TVT_input"].isna()
        & train_feat["linear_tvt_anchor"].notna()
    )
    test_mask = test_feat["id"].isin(sample_submission["id"])

    train_fit = train_feat.loc[train_mask].reset_index(drop=True)
    test_fit = test_feat.loc[test_mask].reset_index(drop=True)
    print("Training post-PS rows:", len(train_fit))
    print("Test rows to predict:", len(test_fit))

    print("\nBuilding spatial OOF prior")
    train_fit, test_fit = fit_predict_spatial_prior(train_fit, test_fit, train_fit["well_id"], cfg)

    features = feature_columns(train_fit, test_fit)
    print("\nFeature count:", len(features))
    print("First 50 features:", features[:50])

    print("\nBaselines")
    print("Anchor RMSE:", rmse(train_fit["TVT"], train_fit["linear_tvt_anchor"]))
    print("Spatial prior RMSE:", rmse(train_fit["TVT"], train_fit["linear_tvt_anchor"] + train_fit["spatial_resid_idw"]))
    print("Typewell best RMSE:", rmse(train_fit["TVT"], train_fit["tw_best_tvt_wide"].fillna(train_fit["linear_tvt_anchor"])))

    print("\nTraining CV models")
    train_fit, test_fit = train_cv_models(train_fit, test_fit, features, cfg)

    print("\nPostprocessing candidates")
    train_fit, test_fit, candidate_cols = add_postprocessed_candidates(train_fit, test_fit, typewells, cfg)

    weights, blend_cols = optimize_blend(train_fit, candidate_cols, cfg)
    train_fit["pred_blend"] = apply_blend(train_fit, weights, blend_cols)
    test_fit["pred_blend"] = apply_blend(test_fit, weights, blend_cols)
    print("Final OOF blend RMSE:", rmse(train_fit["TVT"], train_fit["pred_blend"]))

    cfg.output_path.mkdir(parents=True, exist_ok=True)
    make_submission(sample_submission, test_fit, "pred_blend", cfg.output_path / "submission.csv")
    make_submission(sample_submission, test_fit, "pred_resid_lgbm", cfg.output_path / "submission_raw_lgbm.csv")
    if "pred_viterbi" in test_fit.columns:
        make_submission(sample_submission, test_fit, "pred_viterbi", cfg.output_path / "submission_viterbi.csv")

    # Keep diagnostics for quick notebook review.
    diag_cols = ["id", "well_id", "linear_tvt_anchor", "pred_resid_lgbm", "pred_direct_lgbm", "pred_blend"]
    if "pred_viterbi" in test_fit.columns:
        diag_cols.append("pred_viterbi")
    test_fit[diag_cols].to_csv(cfg.output_path / "test_prediction_diagnostics.csv", index=False)
    print("Saved diagnostics")


if __name__ == "__main__":
    main()
